# Search engine with Tools and Agents 

In [47]:
## arxiv search for papers with "deep learning" in the title, sorted by relevance
## Tools creation
from langchain_community.tools import ArxivQueryRun,WikipediaQueryRun
from langchain_community.utilities import WikipediaAPIWrapper,ArxivAPIWrapper



In [48]:
import wikipedia
wikipedia.set_user_agent("LangchainApp/1.0 (learning project)")

## Used in build tool of Wikipedia. 
api_wrapper_wiki = WikipediaAPIWrapper(top_k_results=1,doc_content_chars_max=1000)
tool_wiki = WikipediaQueryRun(api_wrapper=api_wrapper_wiki)
tool_wiki.name

'wikipedia'

In [49]:
api_wrapper_arxiv = ArxivAPIWrapper(top_k_results=2,doc_content_chars_max=1000)
tool_arxiv = ArxivQueryRun(api_wrapper=api_wrapper_arxiv)
print(tool_arxiv.name)

arxiv


In [50]:
tools=[tool_wiki,tool_arxiv]

In [51]:
##Custom tools [RAG tools]
from langchain_community.document_loaders import WebBaseLoader
from langchain_community.vectorstores import FAISS
from langchain_openai.embeddings import OpenAIEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [52]:
loader=WebBaseLoader("https://docs.smith.langchain.com/")
docs=loader.load()
text_splitter=RecursiveCharacterTextSplitter(chunk_size=1000,chunk_overlap=0).split_documents(docs)
vectordb=FAISS.from_documents(text_splitter,OpenAIEmbeddings())
retriever=vectordb.as_retriever()
retriever

VectorStoreRetriever(tags=['FAISS', 'OpenAIEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x1760a8b80>, search_kwargs={})

In [53]:
from langchain_core.tools.retriever import create_retriever_tool


retriever_tool = create_retriever_tool(
    retriever,
    name="langsmith_search",
    description="Search for information about LangSmith documentation"
)
retriever_tool.name

'langsmith_search'

In [54]:
tools=[tool_wiki,tool_arxiv,retriever_tool]

In [55]:
tools

[WikipediaQueryRun(api_wrapper=WikipediaAPIWrapper(wiki_client=<module 'wikipedia' from '/Users/ankitkumar/Downloads/GenAI/Langchain/venv/lib/python3.10/site-packages/wikipedia/__init__.py'>, top_k_results=1, lang='en', load_all_available_meta=False, doc_content_chars_max=1000)),
 ArxivQueryRun(api_wrapper=ArxivAPIWrapper(arxiv_search=<class 'arxiv.arxiv.Search'>, arxiv_exceptions=(<class 'arxiv.arxiv.ArxivError'>, <class 'arxiv.arxiv.UnexpectedEmptyPageError'>, <class 'arxiv.arxiv.HTTPError'>), top_k_results=2, ARXIV_MAX_QUERY_LENGTH=300, continue_on_failure=False, load_max_docs=100, load_all_available_meta=False, doc_content_chars_max=1000)),
 StructuredTool(name='langsmith_search', description='Search for information about LangSmith documentation', args_schema=<class 'langchain_core.tools.retriever.RetrieverInput'>, func=<function create_retriever_tool.<locals>.func at 0x175dd0d30>, coroutine=<function create_retriever_tool.<locals>.afunc at 0x175dd1240>)]

In [56]:
## Run all these tools with agents and LLM models. 
import os

from dotenv import load_dotenv
from langchain_groq import ChatGroq
from langchain_openai.embeddings import OpenAIEmbeddings
load_dotenv()
os.environ.setdefault("USER_AGENT", "LangchainApp/1.0")
groq_api_key = os.getenv("GROQ_API_KEY")
openai_api_key = os.getenv("OPENAI_API_KEY")

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

llm = ChatGroq(model="openai/gpt-oss-120b", groq_api_key=groq_api_key)



In [57]:
## Prompt template for agent.
from langsmith.client import Client

client = Client()
prompt = client.pull_prompt("hwchase17/openai-functions-agent", dangerously_pull_public_prompt=True)
prompt.messages


[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=[], input_types={}, partial_variables={}, template='You are a helpful assistant'), additional_kwargs={}),
 MessagesPlaceholder(variable_name='chat_history', optional=True),
 HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['input'], input_types={}, partial_variables={}, template='{input}'), additional_kwargs={}),
 MessagesPlaceholder(variable_name='agent_scratchpad')]

In [58]:
# # AGENT EXECUTION
from langchain_classic.agents import create_openai_tools_agent
agent = create_openai_tools_agent(llm=llm, tools=tools, prompt=prompt)
agent

RunnableAssign(mapper={
  agent_scratchpad: RunnableLambda(lambda x: format_to_openai_tool_messages(x['intermediate_steps']))
})
| ChatPromptTemplate(input_variables=['agent_scratchpad', 'input'], optional_variables=['chat_history'], input_types={'chat_history': list[typing.Annotated[typing.Union[typing.Annotated[langchain_core.messages.ai.AIMessage, Tag(tag='ai')], typing.Annotated[langchain_core.messages.human.HumanMessage, Tag(tag='human')], typing.Annotated[langchain_core.messages.chat.ChatMessage, Tag(tag='chat')], typing.Annotated[langchain_core.messages.system.SystemMessage, Tag(tag='system')], typing.Annotated[langchain_core.messages.function.FunctionMessage, Tag(tag='function')], typing.Annotated[langchain_core.messages.tool.ToolMessage, Tag(tag='tool')], typing.Annotated[langchain_core.messages.ai.AIMessageChunk, Tag(tag='AIMessageChunk')], typing.Annotated[langchain_core.messages.human.HumanMessageChunk, Tag(tag='HumanMessageChunk')], typing.Annotated[langchain_core.messages

In [59]:
## Agent execution with a query.
from langchain_classic.agents import AgentExecutor
agent_executor = AgentExecutor(agent=agent, tools=tools, verbose=True)
agent_executor

AgentExecutor(verbose=True, agent=RunnableMultiActionAgent(runnable=RunnableAssign(mapper={
  agent_scratchpad: RunnableLambda(lambda x: format_to_openai_tool_messages(x['intermediate_steps']))
})
| ChatPromptTemplate(input_variables=['agent_scratchpad', 'input'], optional_variables=['chat_history'], input_types={'chat_history': list[typing.Annotated[typing.Union[typing.Annotated[langchain_core.messages.ai.AIMessage, Tag(tag='ai')], typing.Annotated[langchain_core.messages.human.HumanMessage, Tag(tag='human')], typing.Annotated[langchain_core.messages.chat.ChatMessage, Tag(tag='chat')], typing.Annotated[langchain_core.messages.system.SystemMessage, Tag(tag='system')], typing.Annotated[langchain_core.messages.function.FunctionMessage, Tag(tag='function')], typing.Annotated[langchain_core.messages.tool.ToolMessage, Tag(tag='tool')], typing.Annotated[langchain_core.messages.ai.AIMessageChunk, Tag(tag='AIMessageChunk')], typing.Annotated[langchain_core.messages.human.HumanMessageChunk, Tag

In [60]:
agent_executor.invoke({"input": "What is LangSmith?"})




> Entering new AgentExecutor chain...

Invoking: `langsmith_search` with `{'query': 'LangSmith'}`


LangSmith docs - Docs by LangChainSkip to main contentDocs by LangChain home pageLangSmithSearch...⌘KAsk AIGitHubTry LangSmithTry LangSmithSearch...NavigationLangSmith docsGet startedObservabilityEvaluationPrompt engineeringAgent deploymentPlatform setupReferenceOverviewCreate an account and API keyProfile configurationIntegrationsPlansEnterprise featuresLLM GatewayPrivate betaOverviewSpend policiesPII and secrets redactionAccount administrationOverviewWorkspace setupUsers & access controlBilling & usageManage organizations using the APIAudit logsToolsPolly AI assistantCLISkillsSandboxesAdditional resourcesData & complianceFAQLangSmith statusLangSmith docsCopy pageCopy pageDocumentation IndexFetch the complete documentation index at: https://docs.langchain.com/llms.txtUse this file to discover all available pages before exploring further.LangSmith is a framework-agnostic platform for b

{'input': 'What is LangSmith?',
 'output': '**LangSmith** is a\u202fframework‑agnostic platform created by the LangChain team for the entire lifecycle of LLM‑powered applications and AI agents.  \n\nIt bundles together the tools you need to **build, debug, evaluate, and deploy** generative‑AI systems, whether you’re using LangChain, LangGraph, or any other library.\n\n---\n\n### Core capabilities\n\n| Area | What LangSmith provides |\n|------|--------------------------|\n| **Observability** | Automatic tracing of every request, step, and LLM call. You can view detailed execution graphs, inspect inputs/outputs, and quickly locate bugs or performance bottlenecks. |\n| **Evaluation** | Built‑in metrics and evaluation pipelines to measure output quality over time, run custom tests, and keep a history of experiment results. |\n| **Prompt engineering** | Versioned prompt storage, collaborative editing, and “prompt‑as‑code” tools so you can iterate on prompts, compare revisions, and ship impr

In [62]:
agent_executor.invoke({"input": "Can you tell me what machine learning is? Let's check the Wikipedia for this."})



> Entering new AgentExecutor chain...

Invoking: `wikipedia` with `{'query': 'Machine learning'}`


Page: Machine learning
Summary: Machine learning (ML) is a field of study in artificial intelligence concerned with the development and study of statistical algorithms that can learn from data and generalize to unseen data, and thus perform tasks without being explicitly programmed. Advances in the field of deep learning have allowed neural networks, a class of statistical algorithms, to surpass many previous machine learning approaches in performance.
Statistics and mathematical optimisation methods compose the foundations of machine learning. Data mining is a related field of study, focusing on exploratory data analysis (EDA) through unsupervised learning.
From a theoretical viewpoint, probably approximately correct learning provides a mathematical and statistical framework for describing machine learning. Most traditional machine learning and deep learning algorithms can be describe

{'input': "Can you tell me what machine learning is? Let's check the Wikipedia for this.",
 'output': '**Machine learning (ML)** is a sub‑field of artificial intelligence that focuses on creating and studying algorithms that can **learn from data** and make predictions or decisions on new, unseen data without being explicitly programmed for each specific task.\n\nKey points from Wikipedia:\n\n| Aspect | Description |\n|--------|-------------|\n| **Core Idea** | Develop statistical algorithms that improve their performance as they are exposed to more data. |\n| **Goal** | Generalize from observed examples to unseen situations, enabling tasks such as classification, regression, clustering, and recommendation. |\n| **Foundations** | Built on statistics, probability theory, and mathematical optimisation. |\n| **Relation to Other Fields** | Closely related to **data mining** (exploratory data analysis) and **unsupervised learning**. |\n| **Theoretical Framework** | Often framed as **empiric

In [64]:
agent_executor.invoke({"input": "What are paper 1706.03762 about ?"})



> Entering new AgentExecutor chain...

Invoking: `arxiv` with `{'query': '1706.03762'}`




KeyboardInterrupt: 